In [ ]:
import jax.numpy as jnp
import numpy as np
from fft import *
import jax
from symmetry import *
from phase_retrieval import *
from obj import *
from noise import *
from mle import *
import device
from pies import *
from ll_plotting import *
from ptychography import *
from imaging import make_circular_mask
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
from mle_pty import *
from spotgen import *
from scipy.stats import qmc

jax.config.update("jax_enable_x64", True)
cdtype = jnp.array(1j).dtype
gpu_idx, free_mb, gpu_count = device.setup_device(device_id=0)

In [ ]:
N_crop = 256
N_pad = 32

real_object = False

# Ensure that cameraman is nowhere exactly zero to avoid possible singular behavior (for now)
nonzero_cameraman = cameraman() * 0.9 + 0.1

if real_object:
    # Construct object with cameraman in intensity
    x_obj, obj_support = crop_and_pad(nonzero_cameraman, N_crop=N_crop, N_pad=N_pad)
else:
    # Construct object with cameraman in intensity and mandrill in phase
    x_cameraman = nonzero_cameraman
    # TODO: make estimator statistics more robust to avoid needing to stay away from the 'cut'
    x_mandrill = as_phase_field(mandrill() ** 2 * 0.9 - 0.45)
    x_obj, obj_support = crop_and_pad(
        x_cameraman * x_mandrill, N_crop=N_crop, N_pad=N_pad
    )

x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = create_physical_axes(
    x_obj
)
# Detector NA
NA_det = 0.5
F_pupil = make_circular_mask(k, k_0 * NA_det)


N_compare_crop = N_crop // 2

print(r_extent, k_extent)
# Define linear operator
A = jit(lambda x: F_pupil * fft2_shifted(x))
A_dagger = jit(lambda x: ifft2_shifted(F_pupil * x))
yhat = A(x_obj)
plot_intensity_phase(x_obj, extent=r_extent)
plot_intensity_phase(yhat, extent=k_extent, use_log=True)

# 1. Multi-spot real-space ptychography with slight 2D scanning
Slight translations of the multi-spot pattern using a galvo-mirror should be possible. 

In [ ]:
min_x, max_x, min_y, max_y = r_extent

# Define multi-spot array properties like number of spots, NA of spot and spatial area it covers
L_x, L_y = 28, 28
NA_spot = 0.5
# With detector and spot NA, a bound on the total synthetic aperture NA can be given
NA_syn_opt = NA_det + NA_spot

R_opt = 1 / (2 * NA_syn_opt)  # lambda
resolution = R_opt
frac_x = 0.7
frac_y = 0.7
array_bounds = min_x * frac_x, max_x * frac_x, min_y * frac_y, max_y * frac_y
p_x = (array_bounds[1] - array_bounds[0]) / (L_x - 1)
p_y = (array_bounds[3] - array_bounds[2]) / (L_y - 1)
print(p_x, p_y, resolution)

# Define 25 probe positions using random distribution in +-3 p_x central area
M = 100
r = p_x
delta_r = r * 3.0
seed = 1776
base_key = jax.random.PRNGKey(seed)


probe_type = "uniform"

if probe_type == "sobol":

    def create_R_probes_Sobol(M, delta_r, seed=seed):
        sampler = qmc.Sobol(d=2, scramble=True, seed=seed)
        u = sampler.random_base2(m=int(np.log2(M)))  # (M, 2), balanced Sobol set

        deltas = 2.0 * delta_r * u - delta_r
        return jnp.asarray(deltas)

        # M = 32

    R_probes = create_R_probes_Sobol(M, delta_r, seed)
else:
    R_probes = jax.random.uniform(
        base_key, shape=(M, 2), minval=-delta_r, maxval=delta_r
    )

# Define probe arrays
f_ms = jit(
    lambda R_x, R_y: create_multispot_array_product(
        NA_spot,
        array_bounds[0] + R_x,
        array_bounds[1] + R_x,
        array_bounds[2] + R_y,
        array_bounds[3] + R_y,
        L_x,
        L_y,
        K_X,
        K_Y,
        k,
        k_0,
    )
)

N_photons = 1e9
probes = create_probes(R_probes, f_ms, N_photons, cdtype)

# Generate detector field and plot example frame
ys = A(x_obj * probes)
plot_intensity_phase(ys[-1, :], use_log=True)

probed = analyse_probe(probes, x_obj, r_extent)
plot_R_probes(R_probes, r)

In [ ]:
airy_det = ifft2_shifted(F_pupil)
base_probe = probes[0]
filtered_probe = ifft2_shifted(jnp.abs(fft2_shifted(base_probe))) * airy_det
spectral_support = (fft2_shifted(filtered_probe) > 5).astype(
    jnp.real(filtered_probe).dtype
)
plot_intensity_phase(filtered_probe)
plot_intensity_phase(fft2_shifted(filtered_probe))
plot_intensity_phase(ifft2_shifted(fft2_shifted(base_probe) * F_pupil), extent=r_extent)

In [ ]:
base_probe = create_probes(jnp.array([[0.0], [0.0]]), f_ms, N_photons, cdtype)[0]
plot_intensity_phase(base_probe, label="p", extent=r_extent)
plot_intensity_phase(fft2_shifted(base_probe), extent=k_extent)
probe_spect = fft2_shifted(base_probe)
plot_intensity_phase(probe_spect, extent=k_extent)


def plot_intensity_phase_cust(
    y, extent=None, circle=None, label="P", use_log=False, vertical=False
):
    I = jnp.abs(y) ** 2
    if use_log:
        I = jnp.log(I)
    phi = jnp.angle(y)
    if vertical:
        fig, ax = plt.subplots(2, 1, figsize=(6, 12), constrained_layout=True)
    else:
        fig, ax = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
    im0 = ax[0].imshow(I, extent=extent)
    fig.colorbar(im0, ax=ax[0], fraction=0.045, pad=0.02)
    ax[0].set_title(rf"$|{label}|^2$")

    im1 = ax[1].imshow(phi, extent=extent, vmin=-jnp.pi, vmax=jnp.pi)
    fig.colorbar(im1, ax=ax[1], fraction=0.045, pad=0.02)
    ax[1].set_title(rf"$\angle {label}$")
    if circle is not None:
        ax[0].add_patch(circle, label=r"$NA_{spot}=0.5$")
    plt.legend()
    plt.show()


circle = Circle(
    (0, 0),
    k_0 * 0.5,
    fill=False,
    edgecolor="red",
    linewidth=2.2,
    linestyle="--",
)
plot_intensity_phase(probe_spect, extent=k_extent, label="P", circle=circle)

# Plot approximate spectral support of measurement

In [ ]:
y = jnp.abs(probe_spect) ** 2
if jnp.iscomplexobj(y):
    y = jnp.abs(y)

fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)

im = ax.imshow(np.asarray(y), extent=k_extent)
cbar = plt.colorbar(im, ax=ax, fraction=0.045, pad=0.02)

ax.add_patch(
    Circle(
        (0, 0),
        k_0 * NA_spot,
        fill=False,
        edgecolor="red",
        linewidth=2.2,
        linestyle="--",
    )
)

# Labels / legend
handles = [
    Line2D(
        [0],
        [0],
        color="red",
        lw=2.2,
        linestyle="--",
        label=rf"$NA_{{spot}}={NA_spot}$",
    ),
]

ax.legend(handles=handles, loc="upper right", fontsize=11)
ax.set_aspect("equal")

plt.show()

In [ ]:
plot_intensity_phase(
    fft2_shifted(x_obj) * spectral_support, extent=k_extent, use_log=True
)
x_obj_est = ifft2_shifted(fft2_shifted(x_obj) * spectral_support)
plot_intensity_phase(x_obj_est, extent=r_extent)

In [ ]:
y = spectral_support
if jnp.iscomplexobj(y):
    y = jnp.abs(y)

fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)

im = ax.imshow(np.asarray(y), extent=k_extent)
cbar = plt.colorbar(im, ax=ax, fraction=0.045, pad=0.02)

ax.add_patch(
    Circle(
        (0, 0),
        k_0 * NA_syn_opt,
        fill=False,
        edgecolor="red",
        linewidth=2.2,
        linestyle="--",
    )
)

handles = [
    Line2D(
        [0],
        [0],
        color="red",
        lw=2.2,
        linestyle="--",
        label=rf"$NA_{{spot}} + NA_{{det}} = {NA_syn_opt:g}$",
    ),
]

ax.legend(handles=handles, loc="upper right", fontsize=11)
ax.set_aspect("equal")
plt.show()

In [ ]:
x_est = pie(
    ys,
    probes,
    gamma=0.75,
    # alpha=0.9,
    N_iter=400,
    obj_support=obj_support,
    A=A,
    A_dagger=A_dagger,
)
compare_estimate(x_est * probed, x_obj * probed, N_compare_crop=N_crop)

In [ ]:
x_est = ppie(
    ys, probes, gamma=0.75, N_iter=400, obj_support=obj_support, A=A, A_dagger=A_dagger
)
compare_estimate(x_est * probed, x_obj * probed, N_compare_crop=N_crop)

# 2. Generate measurement realizations and perform MLE refinement

In [ ]:
N_samples = 8
# y_mu, y_data = sample_field(ys, N_photons=N_photons, batch_dim=N_samples)
_, y_data = sample_from_mu(jnp.abs(ys) ** 2, batch_dim=N_samples)
for m in range(min(M, 3)):
    plot(jnp.abs(ys[m, :]) ** 2, use_log=True, extent=k_extent)
    plot_intensity_phase(ys[m, :], label="y^m", use_log=True, extent=k_extent)

In [ ]:
N_samples = 8
# y_mu, y_data = sample_field(ys, N_photons=N_photons, batch_dim=N_samples)
_, y_data = sample_from_mu(jnp.abs(ys) ** 2, batch_dim=N_samples)
for m in range(min(M, 3)):
    plot(jnp.abs(y_data[m, 0, :]) ** 2, label=rf"$d^m$", use_log=True)

In [ ]:
def plot_custom(y, extent=None, use_log=False, label="", fontsize=12):
    if use_log:
        y = jnp.log(y)

    plt.figure(figsize=(6, 6), constrained_layout=True)
    im = plt.imshow(y, extent=extent)
    cbar = plt.colorbar(im, fraction=0.045, pad=0.02)
    cbar.set_label(label, fontsize=fontsize)
    plt.show()


for m in range(min(M, 3)):
    plot_custom(jnp.abs(y_data[0, m, :]) ** 2, use_log=True)
    plt.savefig("plot-{m}.png", transparent=True, dpi=300)

In [ ]:
# Was real before, causing reconstruction bugs!
x_pies = jnp.zeros((N_samples,) + x_est.shape, cdtype)
for i in tqdm(range(N_samples)):
    y_complex = jnp.astype(y_data[i, :], cdtype)
    # Already choose the best representative
    x_pies = x_pies.at[i, :].set(
        choose_representative(
            pie(
                y_complex,
                probes,
                gamma=0.75,
                N_iter=500,
                obj_support=obj_support,
                A=A,
                A_dagger=A_dagger,
            ),
            x_obj,
        )
    )
    if i == 0:
        print(y_complex.shape, x_obj.shape)
        print(y_data.dtype, probes.dtype, y_complex.dtype, x_pies.dtype)
        plot_intensity_phase(x_pies[i, :])

In [ ]:
# x_model_ml_first_order, lls_first_order = mle_ptycho_first_order_consensus(
#     x_pies,
#     probes,
#     y_data,
#     obj_support,
#     N_iter=1000,
#     # TODO: find better way to set alpha, like tracking convergence or considering bias/std of estimator
#     alpha=1e-4,
#     epsilon=1e-10,
#     track_ll=True,
#     A=A,
#     A_dagger=A_dagger,
# )

In [ ]:
# norm = construct_norm_ll(lls_first_order)
# plot_ll(lls_first_order, norm)

In [ ]:
x_model_ml_second_order, lls_second_order = mle_ptycho_second_order_consensus(
    x_pies,
    probes,
    y_data,
    obj_support,
    # TODO: find better way to set alpha, like tracking convergence or considering bias/std of estimator
    alpha=1.0,
    N_iter=1000,
    epsilon=1e-10,
    track_ll=True,
    A=A,
    A_dagger=A_dagger,
)

In [ ]:
plot_ll(lls_second_order, norm)

In [ ]:
compare_lls(lls_second_order, lls_first_order)

In [ ]:
compare_lls(
    lls_second_order,
    lls_first_order,
    geometric_mean=True,
    labels=["Second order", "First order"],
)

In [ ]:
plot_intensity_phase(fft2_shifted(x_model_ml_first_order[-1]), use_log=False)
plot_intensity_phase(x_model_ml_first_order[-1], use_log=False)

In [ ]:
lim = lambda x: center_crop(x, 256)
plot_intensity_phase(fft2_shifted(x_model_ml_second_order[-1]), use_log=True)
plot_intensity_phase(x_model_ml_second_order[-1], use_log=False)
plot_intensity_phase(lim(x_model_ml_second_order[-1]), use_log=False)

# Estimator statistics
First evaluates plain PIE output, then PIE followed by ML stage.

In [ ]:
from estimator import evaluate_estimator

evaluate_estimator(lim(x_pies), lim(x_obj), lim(obj_support), label="PIE")

In [ ]:
# plot_intensity_phase(fft2_shifted(x_model_ml_first_order[-1]), use_log=True)

In [ ]:
# evaluate_estimator(
#     lim(x_model_ml_first_order), lim(x_obj), lim(obj_support), label="PIE+ML 1st"
# )

In [ ]:
# evaluate_estimator(
#     lim(x_model_ml_first_order), lim(x_obj_est), lim(obj_support), label="PIE+ML 1st"
# )

In [ ]:
evaluate_estimator(
    lim(x_model_ml_second_order), lim(x_obj), lim(obj_support), label="PIE+ML 2nd"
)

In [ ]:
evaluate_estimator(
    lim(x_model_ml_second_order), lim(x_obj_est), lim(obj_support), label="PIE+ML 2nd"
)